In [236]:
import torch
from math import sqrt

In [97]:
with open(
    "Obras_completas_ilustradas_Federico_Garcia_Lorca_djvu.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()


In [98]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [99]:
text[:8]
encoded_text = torch.tensor(encode(text), dtype = torch.long)

In [255]:
train_size = int(len(text)*0.9)
train_data = encoded_text[:train_size]
test_data = encoded_text[train_size:]
block_size = 8
batch_size = 1
torch.manual_seed(0)

def get_batch(split):
    # generate a small batch of data of inputs x and targets y

    ix = torch.randint(len(split) - block_size, (batch_size,))
    x = torch.stack([split[i:i+block_size] for i in ix])
    y = torch.stack([split[i+1:i+block_size+1] for i in ix])
    return x, y

In [256]:
first_batch = get_batch(train_data)
first_batch[1]

tensor([[36, 47, 28,  1,  1, 46, 28, 41]])

In [257]:
vocab_size = len(chars)
d_model = 10
embedding = torch.nn.Embedding(vocab_size, d_model)

In [ ]:
class Attention(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.
        

In [343]:
positional_encoding = torch.empty(1,8,10)

In [346]:
positional_encoding[:,1]

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [363]:
class positionalEncoding():

    def __init__(self, batch_size, block_size, d_model):

        self.positional_encoding = torch.tensor((batch_size, block_size, d_model))
    def encode_vector():
        for pos in range(block_size):
            for i in range(0, d_model, 2):
                denom = 10000 ** (i / d_model)
                self.positional_encoding[:, pos, i]     = math.sin(pos / denom)
                self.positional_encoding[:, pos, i + 1] = math.cos(pos / denom)
                
        return None

In [379]:
d_k = 5

#class head(torch.nn.Module):

class decoder_only_transformer(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoding = torch.zeros(batch_size, block_size, d_model)
        self.W_q = torch.nn.Linear(d_model, d_k)
        self.W_k = torch.nn.Linear(d_model, d_k)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.ln_1 = torch.nn.LayerNorm(d_model)
        self.ln_2 = torch.nn.LayerNorm(d_model)
        self.W_ff1 = torch.nn.Linear(d_model, 2*d_model)
        self.W_ff2 = torch.nn.Linear(2*d_model, d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)

    def encode_vector(self, batch_size, block_size, d_model):
            
            positional_encoding = torch.zeros(batch_size, block_size, d_model)
            
            for pos in range(block_size):
                for i in range(0, d_model, 2):
                    denom = 10000 ** (i / d_model)
                    positional_encoding[:, pos, i]     = math.sin(pos / denom)
                    positional_encoding[:, pos, i + 1] = math.cos(pos / denom)
                    
            return positional_encoding
    
    def forward(self, vec_ins):
        X_e = self.embedding(vec_ins)
        self.pos_encoding = self.encode_vector(batch_size, block_size, d_model)
        X_e = X_e + self.pos_encoding
        Q = self.W_q(X_e) #(B,T,d_k)
        K = self.W_k(X_e) #(B,T,d_k)
        V = self.W_v(X_e) #(B,T,d_model)
        print(Q.shape, K.shape, V.shape, Q.T.shape)
        QV = Q @ K.transpose(1,2)
        QV_scale = QV/sqrt(d_k) #(B,T,T)
        QV_softmax = torch.nn.functional.softmax(QV_scale, dim = 2) #(B,T,T)
        added_weights = QV_softmax @ V
        norm_res_att = self.ln_1(X_e + added_weights)
        linear_1 = self.W_ff1(torch.nn.functional.gelu(norm_res_att))
        linear_2 = self.W_ff2(linear_1)
        norm_linear = self.ln_2(linear_2 + norm_res_att)
        logits = self.linear(norm_linear)
        return logits


In [380]:
my_first_model = decoder_only_transformer()
my_first_model(first_batch[0])

torch.Size([1, 8, 5]) torch.Size([1, 8, 5]) torch.Size([1, 8, 10]) torch.Size([5, 8, 1])


tensor([[[-3.2148e-01,  4.7412e-01, -6.0079e-01,  3.1237e-01, -1.2636e+00,
          -1.1171e+00,  6.2352e-04, -2.0338e-01, -2.7352e-01, -2.1834e-02,
          -4.1751e-01,  8.7854e-01,  3.5769e-01, -8.2780e-01,  1.0035e+00,
          -6.8042e-02,  3.3550e-01,  1.1920e+00,  1.1717e-01,  8.0305e-01,
          -4.6226e-02, -4.4120e-01,  1.1289e-02,  1.1169e+00, -7.2324e-01,
           5.7826e-01, -3.0295e-01,  1.2477e-01, -9.0426e-01, -1.0042e-01,
           4.9039e-01, -7.5457e-01, -1.3481e+00, -5.2258e-01,  1.2765e-01,
           4.7351e-01, -2.3772e-01, -1.1268e+00, -4.2724e-01,  2.8764e-01,
          -8.2978e-01, -5.2803e-02, -1.4013e-01, -4.4807e-01,  3.6350e-01,
          -4.6345e-01, -1.1975e+00, -3.2335e-01,  3.0618e-02, -1.1439e+00,
          -7.2821e-01,  2.7704e-01, -7.0198e-02,  3.6348e-01,  1.1042e+00,
           7.6859e-01,  1.3070e+00, -1.5633e-01, -7.4221e-01,  6.4168e-01,
           5.2816e-01, -5.1078e-01,  2.6158e-02,  7.0516e-01, -3.2880e-01,
           6.3317e-01,  1

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(first_batch[0][0], first_batch[1][0])
print(f'Total loss for this batch: {loss.item()}')

In [ ]:
vocab_size = len(chars)
embedding_dim = 10
embedding = torch.nn.Embedding(vocab_size, embedding_dim)